# Phase 4.4: export the real Llama-3.2-1B .pte (prefill + decode, windowed attention)

Runs `export_llama.py --real-weights` -- the exact same script already validated locally with
random-init weights (16-layer model, `windowed_sdpa_kv_cache` wired into every layer via a map
arm, survives `torch.export -> to_edge -> to_executorch` as two methods: `prefill` (fixed
shape, `start_pos=0`) and `decode` (`Tn=1`, dynamic `start_pos` for a real generate loop).
This run just swaps in the real pretrained checkpoint so the resulting `.pte` is the actual
artifact Phase 4.5 pushes to the phone.

Needs a HF token with access to the gated `meta-llama/Llama-3.2-1B` repo. A T4 runtime is
enough (export is CPU-bound tracing; T4 just gives comfortable RAM headroom for the ~4.9GB
fp32 model).

In [ ]:
!pip install -q torch==2.12.0 torchvision torchaudio executorch==1.3.1 transformers pybind11 huggingface_hub hf_transfer
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"  # parallel Rust downloader -- default huggingface_hub
                                                 # downloader is single-connection and can crawl on
                                                 # shared free-tier VM bandwidth

In [ ]:
from huggingface_hub import login
login()  # paste an HF token with access to meta-llama/Llama-3.2-1B when prompted

In [ ]:
!rm -rf /content/windowed-attention-cpu
!git clone --depth 1 https://github.com/parzi-val/windowed-attention-cpu.git /content/windowed-attention-cpu
%cd /content/windowed-attention-cpu/executorch/phase4_llama

In [ ]:
# Build the pybind kernel used for export_llama.py's eager sanity check (the actual .pte doesn't
# embed this -- it's pure framework-free C++, no ExecuTorch dependency, so a plain g++ compile
# against the Phase 3 kernel source is enough; no need for the full ExecuTorch/NDK build here.
import sysconfig
ext_suffix = sysconfig.get_config_var("EXT_SUFFIX")

!mkdir -p ../phase3_kv_cache/build
!g++ -O2 -shared -std=c++17 -fPIC $(python3 -m pybind11 --includes) \
    ../phase3_kv_cache/windowed_sdpa_kv_cache_kernel.cpp \
    ../phase3_kv_cache/windowed_sdpa_kv_cache_pybind.cpp \
    -o ../phase3_kv_cache/build/windowed_sdpa_kv_cache_pybind{ext_suffix}
!ls ../phase3_kv_cache/build/

In [ ]:
!python export_llama.py --arm map50 --real-weights

In [ ]:
from google.colab import files
files.download("llama_3.2_1b_map50.pte")